# 10 — exp06-style direction build, per model (no downstream evals)

Streamlined fork of `notebooks/06_structural_authority.ipynb`. Same direction-fitting recipe as exp06 (matched-baseline S/U bundles, model-generated `R_S` prefilled symmetrically, MM + pca_diff + pca_center) but **strips the FAKE / NONE-conflict / intervention pipelines** that exp06 layered on top — those were the buggy parts (layer-selection bias toward an artifact at `response_last` L29) and they're not what notebook 07/09 consume.

Output is a single `directions.npz` per model in **exp08-compatible format**, so `notebooks/07_steering_authority.ipynb` and `notebooks/09_mc_logprob_steering.ipynb` can load it just by pointing their path. Per-model output goes to `<OUT_ROOT>/<slug>/directions.npz` + `manifest.json`.

**Why this notebook exists**: exp09 showed exp06's directions steer ~3× harder than exp08's on the MC logprob readout. The lever is the response source — exp06 prefills the *target model's own* greedy completion symmetrically into both S and U arms, while exp08 reused third-party gold responses. To get exp06-strength directions for Llama / Mistral / Gemma / Phi we need to re-run the model-generated-prefill step per model.

**Runtime requirement**: vLLM. mps won't work. Run on an A100 / H100 / similar CUDA box with ≥40 GB VRAM for 7–9B models. Phase 1 (vLLM gen) and Phase 2 (HF activation extraction) run sequentially; vLLM is freed before HF loads. Expected total ~15–30 min on A100 for a 4–8B model with 1k rows × 2 instructions.

Pipeline:
1. Load 1k stratified conflict rows → expand to 2000 single-instruction items.
2. vLLM greedy-generate `R_S` per item under the S condition (`system = baseline + instruction`).
3. Free vLLM. Load HF model with hooks.
4. For each item, build `S_prefill` and `U_prefill` bundles (skip `S_free` / `U_free` — they were exp06's prefill→free transfer-val controls; not needed for direction fitting).
5. Extract activations at `response_first / response_mid / response_last` × all layers.
6. Per (position × layer): fit MM + pca_diff + pca_center, sign-fix (`+direction → toward S`), save.
7. Write `directions.npz` and `manifest.json`.

In [ ]:
# Install (Colab / fresh GPU box). Comment out if running from a working clone.
# !pip install -q git+https://github.com/ChuloIva/Mech_spoof.git
# !pip install -q vllm

In [ ]:
# (Colab only) auth + Drive mount.
# import os
# from google.colab import drive, userdata
# try:    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
# except Exception: pass
# drive.mount('/content/drive')

In [ ]:
# ---- Config ----
from pathlib import Path
from mech_spoof.configs import MODEL_CONFIGS, DATA_DIR

MODEL_KEY     = 'qwen'           # qwen | llama3 | mistral | gemma | phi3 | gemma_small | ...
MAX_ROWS      = 1000             # rows from conflict_pairs_stratified_1k.jsonl  (each → 2 items)
BASELINE_SYS  = 'You are a helpful assistant.'
FREE_GEN_TOK  = 128              # vLLM max_new_tokens for R_S
EXTRACT_BSZ   = 4                # HF activation extraction batch size (raise on bigger GPUs)
EXTRACT_MAXL  = 2048             # truncation length during extraction
SEED          = 42

# Where to write per-model outputs.
# On Colab: OUT_ROOT = Path('/content/drive/MyDrive/mech_spoof_results/exp06_directions')
OUT_ROOT      = Path.cwd().parent / 'exp06_directions_v2'

cfg     = MODEL_CONFIGS[MODEL_KEY]
OUT_DIR = OUT_ROOT / cfg.slug
OUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_PATH = DATA_DIR / 'conflict_pairs_stratified_1k.jsonl'
assert SOURCE_PATH.exists(), f'expected stratified-1k source at {SOURCE_PATH}'

print(f'model     : {MODEL_KEY}  ({cfg.hf_id})')
print(f'slug      : {cfg.slug}')
print(f'out_dir   : {OUT_DIR}')
print(f'source    : {SOURCE_PATH}  (max_rows={MAX_ROWS})')

## Phase 1 — items + vLLM free-gen of `R_S`

Each row contributes one item per instruction (s_instruction, u_instruction). For each item we generate `R_S = greedy(model | system='{baseline}\n{instruction}', user='{original_prompt}')`. That same `R_S` is later prefilled symmetrically into both the S and U arms, so the only difference between the arms is the role placement of the instruction.

Resumable: if `free_responses.jsonl` already exists in `OUT_DIR` we skip the vLLM call.

In [ ]:
import json
from mech_spoof.experiments.exp6_structural_authority import (
    _load_instruction_pool, _generate_responses_vllm, _build_four_bundles,
)

items = _load_instruction_pool(
    source_path=SOURCE_PATH, max_rows=MAX_ROWS, instruction_choices=('s', 'u'),
)
print(f'items: {len(items)}  (rows={MAX_ROWS}, 2 instructions each)')

gen_path = OUT_DIR / 'free_responses.jsonl'
if gen_path.exists():
    with open(gen_path) as f:
        cached = [json.loads(line) for line in f]
    if len(cached) != len(items):
        raise RuntimeError(f'cached responses ({len(cached)}) != items ({len(items)}); '
                           f'delete {gen_path} to regenerate')
    responses = [c['response'] for c in cached]
    print(f'resumed: {len(responses)} responses from {gen_path}')
else:
    responses = _generate_responses_vllm(
        items=items, vllm_model_id=cfg.hf_id, baseline_system=BASELINE_SYS,
        max_tokens=FREE_GEN_TOK, seed=SEED, free_after=True,
    )
    with open(gen_path, 'w') as f:
        for it, r in zip(items, responses):
            f.write(json.dumps({
                'row_idx': it.row_idx, 'which': it.which,
                'instruction': it.instruction[:200], 'response': r,
            }) + '\n')
    print(f'wrote: {gen_path}  ({len(responses)} responses)')

print('sample R_S:', responses[0][:160].replace('\n', ' '))

## Phase 2 — activation extraction (S_prefill + U_prefill only)

vLLM is freed by `_generate_responses_vllm(free_after=True)`. Now load the HF model with hooks. For each item we build the S_prefill and U_prefill bundles — same `R_S` prefilled into both — and extract residuals at `response_first / response_mid / response_last` for every layer.

Skipping `S_free / U_free` extraction (which exp06 needed for the prefill→free transfer-val numbers). For direction fitting only, we don't need them.

In [ ]:
from mech_spoof.models import load_model
from mech_spoof.activations import extract_multi_position_with_ppl_batched

POSITIONS = ('response_first', 'response_mid', 'response_last')

loaded = load_model(MODEL_KEY)
print(f'device={loaded.device}  n_layers={loaded.n_layers}  d_model={loaded.d_model}')
supports_thinking = getattr(loaded.template, '_supports_enable_thinking', False)

bundles_S, bundles_U = [], []
for it, resp in zip(items, responses):
    four = _build_four_bundles(
        tokenizer=loaded.tokenizer, item=it, baseline_system=BASELINE_SYS,
        response_S=resp, supports_enable_thinking=supports_thinking,
    )
    bundles_S.append(four['S_prefill'])
    bundles_U.append(four['U_prefill'])
print(f'bundles built: {len(bundles_S)} S_prefill + {len(bundles_U)} U_prefill')

def _extract(bundles, name):
    cache = OUT_DIR / f'act_cache_{name}'
    print(f'extracting {name}  →  {cache}')
    acts, _ppl = extract_multi_position_with_ppl_batched(
        loaded, bundles,
        position_names=POSITIONS,
        batch_size=EXTRACT_BSZ, max_length=EXTRACT_MAXL,
        cache_dir=cache, progress=True,
    )
    print(f'  → {acts.shape}')   # (n_items, n_pos, n_layers, d_model)
    return acts

S_acts = _extract(bundles_S, 'S_prefill')
U_acts = _extract(bundles_U, 'U_prefill')
n_items, n_pos, n_layers, d_model = S_acts.shape
assert U_acts.shape == S_acts.shape
print(f'paired tensors: {S_acts.shape}  positions={POSITIONS}')

## Phase 3 — fit MM + pca_diff + pca_center per (position × layer)

Same recipe as `scripts/compute_pca_directions.py` and the exp08 build, with sign-fix `+direction → toward S` (so `mean_S · d > mean_U · d`). NPZ key format matches exp08 so notebooks 07/09 consume it unchanged.

In [ ]:
import numpy as np
from sklearn.decomposition import PCA

def _fit_pca_diff(h_S, h_U):
    diff = h_S - h_U
    v = PCA(n_components=1, whiten=False).fit(diff).components_[0].astype(np.float32)
    return v / (np.linalg.norm(v) + 1e-8)

def _fit_pca_center(h_S, h_U):
    midpoint = (h_S + h_U) / 2.0
    train = np.concatenate([h_S - midpoint, h_U - midpoint], axis=0)
    v = PCA(n_components=1, whiten=False).fit(train).components_[0].astype(np.float32)
    return v / (np.linalg.norm(v) + 1e-8)

def _sign_fix(v, h_S, h_U):
    proj_S = h_S @ v
    proj_U = h_U @ v
    frac_pos = float(np.mean(proj_S > proj_U))
    if proj_S.mean() < proj_U.mean():
        v = -v; frac_pos = 1.0 - frac_pos
    return v, frac_pos

arrays: dict[str, np.ndarray] = {}
manifest_per_layer: dict = {pos: {} for pos in POSITIONS}

for pi, pos in enumerate(POSITIONS):
    print(f'\n[fit] {pos}')
    for layer in range(n_layers):
        h_S = S_acts[:, pi, layer, :]
        h_U = U_acts[:, pi, layer, :]

        # Mass-mean (raw + unit) — natural scale = ||raw||.
        mu_S = h_S.mean(axis=0); mu_U = h_U.mean(axis=0)
        mm_raw  = (mu_S - mu_U).astype(np.float32)
        mm_norm = float(np.linalg.norm(mm_raw))
        mm_unit = (mm_raw / (mm_norm + 1e-10)) if mm_norm > 1e-10 else np.zeros_like(mm_raw)

        # PCA variants.
        v_diff   = _fit_pca_diff(h_S, h_U)
        v_center = _fit_pca_center(h_S, h_U)

        # Sign-fix all three to S>U convention.
        if mm_unit @ (mu_S - mu_U) < 0:   # MM is built from mu_S - mu_U so already correct;
            mm_unit = -mm_unit; mm_raw = -mm_raw
        v_diff,   frac_diff   = _sign_fix(v_diff,   h_S, h_U)
        v_center, frac_center = _sign_fix(v_center, h_S, h_U)

        cos_diff_mm     = float(v_diff   @ mm_unit)
        cos_center_mm   = float(v_center @ mm_unit)
        cos_diff_center = float(v_diff   @ v_center)

        arrays[f'mm_dir__{pos}__layer_{layer:03d}']         = mm_unit.astype(np.float32)
        arrays[f'mm_raw__{pos}__layer_{layer:03d}']         = mm_raw.astype(np.float32)
        arrays[f'pca_diff_dir__{pos}__layer_{layer:03d}']   = v_diff.astype(np.float32)
        arrays[f'pca_center_dir__{pos}__layer_{layer:03d}'] = v_center.astype(np.float32)

        manifest_per_layer[pos][str(layer)] = {
            'mm_natural_scale':         mm_norm,
            'frac_S_proj_larger_diff':  frac_diff,
            'frac_S_proj_larger_center':frac_center,
            'cos_pca_diff_vs_mm':       cos_diff_mm,
            'cos_pca_center_vs_mm':     cos_center_mm,
            'cos_pca_diff_vs_center':   cos_diff_center,
        }

        if layer in (0, 5, 10, 15, 20, 25, n_layers - 1):
            print(f'  L{layer:02d}  σ={mm_norm:.3f}  cos(diff,MM)={cos_diff_mm:+.3f}  '
                  f'cos(ctr,MM)={cos_center_mm:+.3f}  S>U(diff)={frac_diff:.2f}  '
                  f'S>U(ctr)={frac_center:.2f}')

print(f'\nfit complete: {len(arrays)} arrays  ({len(POSITIONS)} positions × {n_layers} layers × 4 keys)')

In [ ]:
# Save NPZ + manifest.
out_npz = OUT_DIR / 'directions.npz'
np.savez_compressed(out_npz, **arrays)

manifest = {
    'experiment':       'exp10_directions_per_model',
    'recipe':           'exp06-style: matched-baseline S/U + symmetric model-generated R_S prefill',
    'model_key':        MODEL_KEY,
    'hf_id':            cfg.hf_id,
    'slug':             cfg.slug,
    'n_paired':         int(n_items),
    'n_positions':      int(n_pos),
    'n_layers':         int(n_layers),
    'd_model':          int(d_model),
    'positions':        list(POSITIONS),
    'baseline_system':  BASELINE_SYS,
    'source_path':      str(SOURCE_PATH),
    'n_rows_sampled':   int(MAX_ROWS),
    'free_gen_max_tokens': int(FREE_GEN_TOK),
    'seed':             int(SEED),
    'per_layer':        manifest_per_layer,
}
with open(OUT_DIR / 'manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)

print(f'wrote: {out_npz}')
print(f'wrote: {OUT_DIR / "manifest.json"}')

## Phase 4 — sanity printouts

Per-layer natural scale (`||mm_raw||`) and cos(pca_*, MM). For Qwen at this exact recipe we expect σ ≈ 2–3 over the steered band L16..L31 at `response_last` and substantially smaller at `response_first` — different models will differ; what we want to see is that σ rises toward late layers and that pca_* and MM agree (cos ≳ 0.7) where σ is large.

In [ ]:
for pos in POSITIONS:
    sigmas    = [manifest_per_layer[pos][str(l)]['mm_natural_scale']     for l in range(n_layers)]
    cos_diff  = [manifest_per_layer[pos][str(l)]['cos_pca_diff_vs_mm']   for l in range(n_layers)]
    cos_ctr   = [manifest_per_layer[pos][str(l)]['cos_pca_center_vs_mm'] for l in range(n_layers)]
    print(f'\n[{pos}]  median σ over all layers = {float(np.median(sigmas)):.3f}')
    print(f'  layer  σ        cos(diff,MM)  cos(ctr,MM)')
    for l in range(n_layers):
        marker = '  *' if l in (10, 15, 16, 20, 25, n_layers - 1) else ''
        print(f'    L{l:02d}  {sigmas[l]:7.3f}    {cos_diff[l]:+0.3f}        {cos_ctr[l]:+0.3f}{marker}')

In [ ]:
# Optional plot — same shape as exp08's manifest plot.
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for pos in POSITIONS:
    sigmas   = [manifest_per_layer[pos][str(l)]['mm_natural_scale']   for l in range(n_layers)]
    cos_ctr  = [manifest_per_layer[pos][str(l)]['cos_pca_center_vs_mm'] for l in range(n_layers)]
    axes[0].plot(range(n_layers), sigmas,  '-o', label=pos, alpha=0.85)
    axes[1].plot(range(n_layers), cos_ctr, '-o', label=pos, alpha=0.85)
axes[0].set_xlabel('layer'); axes[0].set_ylabel('σ = ||mm_raw||');
axes[0].set_title(f'{MODEL_KEY}: per-layer natural scale')
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)
axes[1].set_xlabel('layer'); axes[1].set_ylabel('cos(pca_center, MM)')
axes[1].set_title(f'{MODEL_KEY}: pca_center ↔ MM agreement')
axes[1].axhline(0, color='gray', linewidth=0.5); axes[1].set_ylim(-0.1, 1.05)
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'sigma_and_cos.png', dpi=120)
plt.show()
print(f'wrote: {OUT_DIR / "sigma_and_cos.png"}')

## How to consume in notebook 09 (or 07)

Point the path at this run's `directions.npz` — the NPZ key format is identical to exp08 so the loader code is unchanged:

```python
# in notebooks/09_mc_logprob_steering.ipynb cell 3
EXP8_NPZ = OUT_ROOT / cfg.slug / 'directions.npz'   # this notebook's output
MODEL_KEY = '<same key>'                              # match the model used here
```

or rename the loader-path variable and treat this as a third source on equal footing with exp06/exp08. Each model will have its own `σ`, so `K = 1.0` is comparable in σ-units across models but maps to a different absolute residual perturbation.